# VS Code Installer
> Install MCP configuration for VS Code

In [ ]:
#| default_exp install.vscode

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from pathlib import Path
from typing import Optional

from nbdev_mcp.install.base import (
    BaseInstaller, InstallResult, MCPServerConfig,
    read_json_config, write_json_config, get_python_path,
    get_vscode_mcp_path, get_vscode_user_dir
)

## VS Code Installer

In [ ]:
#| export
class VSCodeInstaller(BaseInstaller):
    """Installer for VS Code.
    
    Configures MCP in VS Code's mcp.json file.
    Location varies by platform:
    - macOS: ~/Library/Application Support/Code/User/mcp.json
    - Windows: %APPDATA%/Code/User/mcp.json
    - Linux: ~/.config/Code/User/mcp.json
    """
    
    @property
    def name(self) -> str:
        return 'vscode'
    
    @property
    def config_path(self) -> Path:
        return get_vscode_mcp_path()
    
    def is_configured(self) -> bool:
        """Check if nbdev is already configured."""
        config = read_json_config(self.config_path)
        servers = config.get('mcpServers', config.get('servers', {}))
        return 'nbdev' in servers
    
    def do_install(self, server: MCPServerConfig) -> None:
        """Perform the installation."""
        config = read_json_config(self.config_path)
        
        # VS Code uses 'servers' key in mcp.json
        if 'servers' not in config:
            config['servers'] = {}
        
        config['servers'][server.name] = server.to_vscode_format()
        write_json_config(self.config_path, config)
    
    def do_uninstall(self) -> None:
        """Remove the configuration."""
        config = read_json_config(self.config_path)
        
        # Check both 'servers' and 'mcpServers' keys
        for key in ('servers', 'mcpServers'):
            if key in config and 'nbdev' in config[key]:
                del config[key]['nbdev']
        
        write_json_config(self.config_path, config)
    
    def is_vscode_installed(self) -> bool:
        """Check if VS Code user directory exists."""
        return get_vscode_user_dir().exists()

## Convenience Functions

In [ ]:
#| export
def install_vscode(
    python_path: Optional[str] = None,
    force: bool = False,
    dry_run: bool = False
) -> InstallResult:
    """Install nbdev-mcp configuration for VS Code.
    
    Parameters
    ----------
    python_path : str, optional
        Python interpreter path. Defaults to current.
    force : bool
        Overwrite existing configuration.
    dry_run : bool
        Show what would be done without making changes.
    
    Returns
    -------
    InstallResult
        Result of the installation.
    """
    installer = VSCodeInstaller()
    server = MCPServerConfig.for_nbdev(python_path)
    return installer.install(server, force=force, dry_run=dry_run)


def uninstall_vscode(dry_run: bool = False) -> InstallResult:
    """Remove nbdev-mcp configuration from VS Code."""
    installer = VSCodeInstaller()
    return installer.uninstall(dry_run=dry_run)

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()